# 1 · Train the domino detector

**Goal:** find whole domino tiles and their angles with YOLO26-nano OBB.
The second notebook reads each tile's two pip values. This notebook is self-contained:
upload the notebook and your dataset ZIP; no repository clone or helper upload is needed.

**First Colab session**
1. In Google Drive, create `MyDrive/DominoesCalculator` and upload `dominoes-v1.zip` there.
2. Open this notebook in Colab. Choose **Runtime → Change runtime type → GPU** for training.
3. Run setup and preparation. Review the photo groups before creating the split.
4. Set `ACTION = "train"`, rerun configuration and the training/evaluation cells.

The default `prepare` action performs data checks without GPU training. Human review
is intentional: this is a staged experiment, not a blind “Run all” training job.
Keep the final test set untouched until model/threshold choices are settled.

**Storage:** original data, grouping decisions, splits, checkpoints, plots, and exports
stay in Drive. Training images are copied to `/content/domino-work` for faster reads.
Saving the notebook alone does not save its runtime. [Colab storage](https://research.google.com/colaboratory/faq.html).

## Setup
Run installation before importing packages. Setup supports Colab Python 3.12 and 3.13
and shows live installation progress. If setup requests a restart, choose **Runtime →
Restart session**, then rerun setup before continuing. Package versions are pinned;
an environment snapshot is also saved per run.
Export conversion may install additional backend dependencies; restart after such changes if requested.

In [ ]:
"""Embedded in each notebook's setup cell; uses only the standard library."""
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version


NUMPY_REQUIREMENT = "numpy==2.1.3" if sys.version_info >= (3, 13) else "numpy==1.26.4"
TENSORFLOW_REQUIREMENT = "tensorflow==2.20.0" if sys.version_info >= (3, 13) else "tensorflow==2.19.1"
KERAS_REQUIREMENT = "keras==3.10.0" if sys.version_info >= (3, 13) else "keras==3.9.2"


def install_packages(requirements):
    if os.environ.get("DOMINO_SKIP_INSTALL") == "1":
        return
    if not (3, 10) <= sys.version_info[:2] <= (3, 13):
        raise RuntimeError("Use a Python 3.10–3.13 runtime for these pinned packages.")
    if sys.platform == "darwin" and sys.version_info >= (3, 13):
        raise RuntimeError("Use Colab for Python 3.13, or Python 3.12 for local macOS runs.")
    restart_message = "Choose Runtime → Restart session, then rerun setup before continuing."
    if globals().get("_DOMINO_RESTART_REQUIRED"):
        raise RuntimeError(restart_message)

    def installed(name):
        try:
            return package_version(name)
        except PackageNotFoundError:
            return None

    modules = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
               "pillow": "PIL", "opencv-python": "cv2", "torch": "torch",
               "torchvision": "torchvision", "tensorflow": "tensorflow", "keras": "keras",
               "scipy": "scipy", "scikit-learn": "sklearn", "h5py": "h5py"}
    loaded = {name: installed(name) for name, module in modules.items() if module in sys.modules}
    print("Installing packages for Python", sys.version.split()[0], "— progress follows:", flush=True)
    command = [sys.executable, "-u", "-m", "pip", "install",
               "--only-binary=numpy,pandas", *requirements]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()

    # Colab can preload NumPy. Changing files on disk cannot replace its loaded extensions.
    changed = [name for name, previous in loaded.items() if installed(name) != previous]
    if changed:
        globals()["_DOMINO_RESTART_REQUIRED"] = True
        print("Loaded packages changed:", ", ".join(changed), flush=True)
    if returncode:
        raise RuntimeError("Package installation failed. Read the pip error above before continuing. "
                           + (restart_message if changed else ""))
    if changed:
        raise RuntimeError("Installation finished. " + restart_message)
    print("Installation finished. Continue to configuration.", flush=True)


install_packages(["ultralytics==8.4.144", "torch==2.9.0", "torchvision==0.24.0",
                  NUMPY_REQUIREMENT, "pandas==2.2.3", "matplotlib==3.10.1",
                  "pillow==11.1.0", "opencv-python==4.11.0.86"])

### Configuration
`prepare` checks and splits data; `train` starts a new run; `resume` continues its last
checkpoint; `evaluate` loads its best checkpoint; `export` also creates and checks mobile models.
Choose a new `RUN_NAME` for a new experiment. Never overwrite an old run to try new settings.

In [ ]:
DATASET_NAME = "dominoes-v1"
RUN_NAME = "domino-v1-01"
ACTION = "prepare"  # prepare | train | resume | evaluate | export
GROUPS_REVIEWED = False  # Set True only after checking the grouping CSV/contact sheets below.
SEED = 42
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8  # Increase if GPU memory allows.
DETECTION_CONFIDENCE = 0.25  # Tune on validation only, then freeze for the final test.
FINAL_TEST = False
EXPORT_INT8 = True
MAX_EXPORT_MAP_DROP = 0.02  # Absolute mAP50–95 drop allowed relative to FP32/PyTorch.
assert ACTION in {"prepare", "train", "resume", "evaluate", "export"}

### Export runtime dependencies
This runs only for `export`, before importing PyTorch. If you changed to export after
already importing/training in this session, restart the runtime and run from setup with
`ACTION = "export"`. This lets the package resolver keep NumPy/PyTorch compatible.

In [ ]:
if ACTION == "export":
    install_packages([NUMPY_REQUIREMENT, "torch==2.9.0", "torchvision==0.24.0",
                      "litert-torch==0.9.0", "ai-edge-litert==2.1.4", "ai-edge-quantizer==0.6.0"])

In [ ]:
"""Shared helpers embedded into both standalone Colab notebooks by build_notebooks.py."""
from pathlib import Path
from collections import Counter
import csv
import hashlib
import io
import json
import math
import os
import random
import shutil
import sys
import zipfile

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

CLASS_NAMES = [str(i) for i in range(16)]
CROP_VERSION = "long-edge-rgb-128-v1"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def write_csv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_csv(path):
    with Path(path).open(newline="") as f:
        return list(csv.DictReader(f))


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def setup_project():
    """Drive stores durable artifacts; Colab's local disk stores training input."""
    local_override = os.environ.get("DOMINO_ML_ROOT")
    in_colab = not local_override and "google.colab" in sys.modules
    if not local_override:
        try:
            import google.colab  # noqa: F401
            in_colab = True
        except ImportError:
            in_colab = False
    if in_colab:
        from google.colab import drive
        drive.mount("/content/drive")
        project = Path("/content/drive/MyDrive/DominoesCalculator")
        work = Path("/content/domino-work")
    else:
        if local_override:
            project = Path(local_override).expanduser().resolve()
        else:
            candidates = [Path.cwd(), *Path.cwd().parents]
            repo = next((p for p in candidates if (p / "machine_learning").is_dir()), None)
            if repo is None:
                raise RuntimeError("Set DOMINO_ML_ROOT to a local machine_learning folder.")
            project = repo / "machine_learning"
        work = Path(os.environ.get("DOMINO_WORK_DIR", str(project / "data/work")))
    project.mkdir(parents=True, exist_ok=True)
    work.mkdir(parents=True, exist_ok=True)
    return project, work, in_colab


def stage_export(project, work, dataset_name):
    """Accept an extracted export or a ZIP uploaded to the project folder."""
    source = project / "data/raw" / dataset_name
    if not source.is_dir():
        archive = project / f"{dataset_name}.zip"
        if not archive.is_file():
            raise FileNotFoundError(f"Upload {archive.name} to {project}, or place images/ and labels/ in {source}.")
        unpacked = work / "unpacked" / dataset_name
        if unpacked.exists():
            shutil.rmtree(unpacked)  # Only this generated local extraction cache.
        unpacked.mkdir(parents=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                target = (unpacked / member.filename).resolve()
                if not target.is_relative_to(unpacked.resolve()) or (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Archive contains an unsafe path or symbolic link.")
            z.extractall(unpacked)
        candidates = [p.parent for p in unpacked.rglob("classes.txt")
                      if (p.parent / "images").is_dir() and (p.parent / "labels").is_dir()]
        if len(candidates) != 1:
            raise ValueError("Expected exactly one export with classes.txt, images/, and labels/.")
        source.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(candidates[0], source, ignore=shutil.ignore_patterns(".DS_Store", "__MACOSX"))
    # Refresh the generated local cache when source files change; never change the original export.
    local = work / "raw" / dataset_name
    shutil.copytree(source, local, dirs_exist_ok=True, ignore=shutil.ignore_patterns(".DS_Store", "__MACOSX"))
    source_names = {p.relative_to(source) for p in source.rglob("*") if p.is_file() and p.name != ".DS_Store"}
    for p in local.rglob("*"):
        if p.is_file() and p.relative_to(local) not in source_names:
            p.unlink()
    return local


def read_rgb(path):
    # Match browser/phone display orientation before applying annotation coordinates.
    with Image.open(path) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))


def audit_export(root):
    if (root / "classes.txt").read_text().splitlines() != ["domino"]:
        raise ValueError("This pipeline requires exactly one class: domino, ID 0.")
    images = sorted(p for p in (root / "images").iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = {p.stem: p for p in (root / "labels").glob("*.txt")}
    if not images or len({p.stem for p in images}) != len(images):
        raise ValueError("No supported images, or duplicate image stems.")
    if {p.stem for p in images} != set(labels):
        raise ValueError("Every image needs one matching label file; orphan labels are also rejected.")
    records = []
    for path in images:
        rgb = read_rgb(path)
        h, w = rgb.shape[:2]
        boxes = []
        for line_no, line in enumerate(labels[path.stem].read_text().splitlines(), 1):
            if not line.strip():
                continue
            fields = line.split()
            if len(fields) != 9 or fields[0] != "0":
                raise ValueError(f"{path.stem}:{line_no}: expected class 0 and eight OBB corner coordinates.")
            corners = np.asarray(fields[1:], dtype=np.float32).reshape(4, 2)
            if not np.isfinite(corners).all() or (corners < 0).any() or (corners > 1).any():
                raise ValueError(f"{path.stem}:{line_no}: coordinates must be finite and within [0, 1].")
            pixel_corners = corners * [w, h]
            if not cv2.isContourConvex(pixel_corners.astype(np.float32)) or abs(cv2.contourArea(pixel_corners.astype(np.float32))) < 4:
                raise ValueError(f"{path.stem}:{line_no}: invalid or tiny quadrilateral.")
            boxes.append(corners.tolist())
        records.append({"image": path.name, "stem": path.stem, "width": w, "height": h,
                        "image_sha256": sha256(path), "label_sha256": sha256(labels[path.stem]), "boxes": boxes})
    fingerprint = digest_json(records)
    print(f"Validated {len(records)} photos and {sum(len(r['boxes']) for r in records)} domino boxes.")
    return records, fingerprint


def contact_sheet(records, root, start=0, count=12, overlays=False):
    selected = records[start:start + count]
    if not selected:
        return
    fig, axes = plt.subplots(math.ceil(len(selected) / 3), 3, figsize=(15, 3.8 * math.ceil(len(selected) / 3)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, record in zip(axes.flat, selected):
        image = read_rgb(root / "images" / record["image"])
        ax.imshow(image)
        if overlays:
            for box in record["boxes"]:
                points = np.asarray(box) * [record["width"], record["height"]]
                points = np.vstack([points, points[0]])
                ax.plot(points[:, 0], points[:, 1], color="#f6ac48", linewidth=1)
        ax.set_title(record["image"], fontsize=9)
    fig.tight_layout()
    plt.show()


def ensure_groups(records, path):
    if not path.exists():
        # Exact duplicates start together. Repeated arrangements still need human review.
        hashes = {}
        rows = []
        for r in records:
            group = hashes.setdefault(r["image_sha256"], r["stem"])
            rows.append({"image": r["image"], "group": group})
        write_csv(path, rows, ["image", "group"])
    return read_csv(path)


def frozen_split(records, fingerprint, groups_path, output_path, reviewed=False, seed=42):
    rows = read_csv(groups_path)
    if len(rows) != len(records) or {r["image"] for r in rows} != {r["image"] for r in records}:
        raise ValueError("Photo grouping CSV must contain exactly one row per source image.")
    grouping = {r["image"]: r["group"].strip() for r in rows}
    if not all(grouping.values()):
        raise ValueError("Every photo needs a nonempty group.")
    hashes = {}
    for r in records:
        previous = hashes.setdefault(r["image_sha256"], grouping[r["image"]])
        if previous != grouping[r["image"]]:
            raise ValueError("Identical photos must use the same group.")
    signature = digest_json({"fingerprint": fingerprint, "groups": grouping, "seed": seed, "fractions": [0.7, 0.15, 0.15]})
    if output_path.exists():
        result = json.loads(output_path.read_text())
        if result["signature"] != signature:
            raise ValueError("Dataset/groups changed after the split was frozen. Use a new DATASET_NAME and run name; do not mix evaluations.")
        return result
    if not reviewed:
        print("Split not created. Review photo_groups.csv, then set GROUPS_REVIEWED=True and rerun this cell.")
        return None
    groups = sorted(set(grouping.values()))
    if len(groups) < 3:
        raise ValueError("At least three independent arrangement groups are needed for train/val/test.")
    random.Random(seed).shuffle(groups)
    holdout = min(max(1, round(len(groups) * 0.15)), (len(groups) - 1) // 2)
    group_splits = {g: "test" if i < holdout else "val" if i < 2 * holdout else "train" for i, g in enumerate(groups)}
    result = {"signature": signature, "dataset_fingerprint": fingerprint, "seed": seed,
              "photos": [{"image": r["image"], "group": grouping[r["image"]], "split": group_splits[grouping[r["image"]]]} for r in records]}
    write_json(output_path, result)
    return result


def prepare_detector(root, records, split, work):
    target = work / "detector" / split["signature"][:16]
    by_name = {r["image"]: r for r in records}
    for row in split["photos"]:
        record = by_name[row["image"]]
        for kind, src in [("images", root / "images" / row["image"]), ("labels", root / "labels" / f"{record['stem']}.txt")]:
            dest = target / row["split"] / kind / src.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists() or sha256(dest) != sha256(src):
                shutil.copy2(src, dest)
    config = target / "data.yaml"
    config.write_text(f"path: {json.dumps(str(target))}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n  0: domino\n")
    # Exporters may read the YAML's val split for calibration. Keep calibration strictly in train.
    (target / "calibration.yaml").write_text(f"path: {json.dumps(str(target))}\ntrain: train/images\nval: train/images\nnames:\n  0: domino\n")
    return config


def rectify_tile(rgb, corners, half_size=128):
    """Four corners in source pixels -> horizontal RGB tile, left half then right half."""
    p = np.asarray(corners, dtype=np.float32).reshape(4, 2)
    if not np.isfinite(p).all() or not cv2.isContourConvex(p) or abs(cv2.contourArea(p)) < 4:
        raise ValueError("Cannot rectify a degenerate domino detection.")
    # Positive winding in image coordinates, then make a long edge the top edge.
    signed_area = np.sum(p[:, 0] * np.roll(p[:, 1], -1) - p[:, 1] * np.roll(p[:, 0], -1))
    if signed_area < 0:
        p = p[::-1]
    lengths = np.linalg.norm(np.roll(p, -1, axis=0) - p, axis=1)
    p = np.roll(p, -int(np.argmax(lengths)), axis=0).copy()
    target = np.float32([[0, 0], [2 * half_size - 1, 0], [2 * half_size - 1, half_size - 1], [0, half_size - 1]])
    transform = cv2.getPerspectiveTransform(p, target)
    return cv2.warpPerspective(rgb, transform, (2 * half_size, half_size), flags=cv2.INTER_LINEAR,
                               borderMode=cv2.BORDER_CONSTANT, borderValue=(127, 127, 127))


def tile_halves(tile):
    center = tile.shape[1] // 2
    return tile[:, :center].copy(), tile[:, center:].copy()


def score_pairs(pairs, blank_score=0):
    return sum(blank_score if value == 0 else value for pair in pairs if pair is not None for value in pair)


def make_crops(root, records, directory, fingerprint):
    metadata = {"dataset_fingerprint": fingerprint, "crop_version": CROP_VERSION}
    marker = directory / "metadata.json"
    if marker.exists() and json.loads(marker.read_text()) != metadata:
        raise ValueError("Crop dataset changed. Use a new crop directory so existing pip labels remain meaningful.")
    directory.mkdir(parents=True, exist_ok=True)
    write_json(marker, metadata)
    tiles = []
    for record in records:
        rgb = read_rgb(root / "images" / record["image"])
        for index, box in enumerate(record["boxes"]):
            tile_id = f"{record['stem']}_d{index:04d}"
            paths = {side: directory / "halves" / f"{tile_id}_{side}.png" for side in ("a", "b")}
            if not all(p.is_file() for p in paths.values()):
                tile = rectify_tile(rgb, np.asarray(box) * [record["width"], record["height"]])
                for side, half in zip(("a", "b"), tile_halves(tile)):
                    paths[side].parent.mkdir(parents=True, exist_ok=True)
                    Image.fromarray(half).save(paths[side])
            tiles.append({"tile_id": tile_id, "image": record["image"], "box_index": index,
                          "a_path": str(paths["a"].relative_to(directory)), "b_path": str(paths["b"].relative_to(directory))})
    write_csv(directory / "tiles.csv", tiles, ["tile_id", "image", "box_index", "a_path", "b_path"])
    return tiles


def load_pip_labels(path, tiles):
    fields = ["tile_id", "a", "b", "status"]
    if not path.exists():
        write_csv(path, [{"tile_id": t["tile_id"], "a": "", "b": "", "status": "pending"} for t in tiles], fields)
    rows = read_csv(path)
    if len(rows) != len(tiles) or {r["tile_id"] for r in rows} != {t["tile_id"] for t in tiles}:
        raise ValueError("Pip label IDs do not match the crop manifest.")
    for row in rows:
        if row["status"] not in {"pending", "labeled", "rejected"}:
            raise ValueError(f"Invalid labeling status: {row}")
        if row["status"] == "labeled" and (row["a"] not in CLASS_NAMES or row["b"] not in CLASS_NAMES):
            raise ValueError(f"Both halves must have values 0–15: {row}")
    return {r["tile_id"]: r for r in rows}


def classifier_rows(tiles, labels, split):
    photo_split = {r["image"]: r["split"] for r in split["photos"]}
    result = []
    for tile in tiles:
        label = labels[tile["tile_id"]]
        if label["status"] != "labeled":
            continue
        for side in ("a", "b"):
            result.append({"tile_id": tile["tile_id"], "image": tile["image"], "side": side,
                           "path": tile[f"{side}_path"], "value": int(label[side]), "split": photo_split[tile["image"]]})
    return result


def polygon_iou(a, b):
    a, b = np.asarray(a, np.float32), np.asarray(b, np.float32)
    area_a, area_b = abs(cv2.contourArea(a)), abs(cv2.contourArea(b))
    intersection, _ = cv2.intersectConvexConvex(a, b)
    return max(0.0, float(intersection)) / max(area_a + area_b - intersection, 1e-9)


def match_boxes(truth, predicted, threshold=0.5):
    candidates = sorted([(polygon_iou(a, b), i, j) for i, a in enumerate(truth) for j, b in enumerate(predicted)], reverse=True)
    used_truth, used_pred, matches = set(), set(), []
    for overlap, i, j in candidates:
        if overlap >= threshold and i not in used_truth and j not in used_pred:
            used_truth.add(i)
            used_pred.add(j)
            matches.append((i, j, overlap))
    return matches

In [ ]:
from importlib.metadata import version

PROJECT, WORK, IN_COLAB = setup_project()
RAW = stage_export(PROJECT, WORK, DATASET_NAME)
STATE = PROJECT / "data/manifests" / DATASET_NAME
STATE.mkdir(parents=True, exist_ok=True)
RUN = PROJECT / "models/detector" / RUN_NAME
print("Persistent project:", PROJECT)
print("Local training cache:", WORK)
print("Run:", RUN)
print("Action:", ACTION)
print("Python:", sys.version.split()[0])

## Prepare and inspect the dataset
Accepts `.jpeg`, `.jpg`, and `.png`, with class `0 = domino` and eight normalized
corner coordinates per object. Images are opened in their display/EXIF orientation.
Inspect overlays before training; format validity alone does not prove boxes are well placed.

In [ ]:
records, fingerprint = audit_export(RAW)
write_json(STATE / "dataset.json", {"fingerprint": fingerprint, "photos": records})
contact_sheet(records, RAW, start=0, count=6, overlays=True)

### Review repeated arrangements before splitting
The CSV starts with one group per photo; exact file duplicates share a group automatically.
Photos of the **same arrangement**, including alternate angles, belong to the same group.
Review all contact-sheet pages. Edit the table below and rerun it, or edit
`data/manifests/dominoes-v1/photo_groups.csv` in Drive. Keep the image names unchanged.

Use stable short group names such as `wood_arrangement_1`. Group by arrangement/session
when shots are closely related, not merely by which physical domino tiles appear.
If there are fewer than three independent groups, collect more arrangements first.

In [ ]:
GROUPS_PATH = STATE / "photo_groups.csv"
group_rows = ensure_groups(records, GROUPS_PATH)
# Example after viewing your photos: {"a94404f6-IMG_0801.jpeg": "wood_arrangement_1"}
GROUP_OVERRIDES = {}
unknown = set(GROUP_OVERRIDES) - {r["image"] for r in group_rows}
if unknown:
    raise ValueError(f"Unknown image names in GROUP_OVERRIDES: {sorted(unknown)}")
for row in group_rows:
    row["group"] = GROUP_OVERRIDES.get(row["image"], row["group"])
write_csv(GROUPS_PATH, group_rows, ["image", "group"])
display(pd.DataFrame(group_rows))
for start in range(0, len(records), 12):
    contact_sheet(records, RAW, start=start, count=12)

In [ ]:
split = frozen_split(records, fingerprint, GROUPS_PATH, STATE / "split.json", GROUPS_REVIEWED, SEED)
DATA_YAML = None
if split:
    DATA_YAML = prepare_detector(RAW, records, split, WORK)
    counts = pd.DataFrame(split["photos"]).groupby("split").agg(photos=("image", "count"), groups=("group", "nunique"))
    display(counts.reindex(["train", "val", "test"]))
    ax = counts.reindex(["train", "val", "test"])["photos"].plot.bar(color="#34699a", rot=0)
    ax.set(title="Photos in the frozen split", ylabel="Original photos", xlabel="Split", ylim=(0, None))
    plt.tight_layout()
    plt.show()
    print("Frozen split:", STATE / "split.json")
    print("Dataset YAML:", DATA_YAML)

## Train or resume
Uses `yolo26n-obb.pt`, which accepts the Label Studio OBB export. The “YOLOv8” export name
describes the annotation format; it does not require using a YOLOv8 model.
[OBB training and results](https://docs.ultralytics.com/tasks/obb/).

Checkpoints are written to Drive each epoch. For an interrupted run, reconnect, rerun setup
and preparation, keep the same `RUN_NAME`, and choose `resume`. For a completed experiment,
choose `evaluate` or a new run name. Resume restores the model/optimizer from `last.pt`.
Set a realistic epoch budget and inspect validation; no fixed training time or accuracy is promised.

In [ ]:
detector = None
if ACTION != "prepare":
    if split is None:
        raise RuntimeError("Review photo groups and create the split before training/evaluation.")
    import torch
    from ultralytics import YOLO
    DEVICE = 0 if torch.cuda.is_available() else "cpu"
    if ACTION in {"train", "resume"} and DEVICE == "cpu" and not os.environ.get("DOMINO_ALLOW_CPU"):
        raise RuntimeError("Select a Colab GPU runtime, then rerun setup. CPU training is disabled by default.")
    contract = {"split_signature": split["signature"], "model": "yolo26n-obb.pt", "image_size": IMAGE_SIZE, "seed": SEED}
    if ACTION == "train":
        if RUN.exists():
            raise FileExistsError("Run already exists. Use resume/evaluate or choose a new RUN_NAME.")
        RUN.mkdir(parents=True)
        write_json(RUN / "dataset_contract.json", contract)
        (RUN / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
        detector = YOLO("yolo26n-obb.pt")
        detector.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
                       seed=SEED, deterministic=True, patience=20, device=DEVICE,
                       project=str(RUN.parent), name=RUN.name, exist_ok=True, save=True, plots=True)
    else:
        if not (RUN / "dataset_contract.json").is_file() or json.loads((RUN / "dataset_contract.json").read_text()) != contract:
            raise ValueError("Run configuration does not match this dataset/model/image size/seed.")
        if ACTION == "resume":
            checkpoint = RUN / "weights/last.pt"
            if not checkpoint.exists():
                raise FileNotFoundError("No last.pt exists yet. Choose a new run name to restart.")
            detector = YOLO(str(checkpoint))
            detector.train(resume=True, device=DEVICE)
    best = RUN / "weights/best.pt"
    if not best.is_file():
        raise FileNotFoundError(f"No trained checkpoint found: {best}")
    detector = YOLO(str(best))
    print("Using this run's best checkpoint:", best)
else:
    print("Preparation complete. Set ACTION='train' when the grouping and overlays are reviewed.")

## Check detection quality
Validation mAP measures overlap and ranking, while missed/extra-tile counts show failures
that matter for scoring. Inspect the examples as well as the numbers. Diagnostics below use
one-to-one matching at IoU 0.5; official mAP is calculated separately by Ultralytics.

In [ ]:
if detector is not None:
    validation = detector.val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE,
                              device=DEVICE, project=str(RUN / "checks"), name="val", exist_ok=True, plots=True)
    val_scores = {"map50": float(validation.box.map50), "map50_95": float(validation.box.map)}
    write_json(RUN / "validation.json", val_scores)
    display(pd.DataFrame([val_scores]))
    training_csv = RUN / "results.csv"
    if training_csv.exists():
        history = pd.read_csv(training_csv)
        history.columns = history.columns.str.strip()
        metric_columns = [c for c in history if "mAP" in c]
        loss_columns = [c for c in history if "loss" in c]
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        history.plot(x="epoch", y=loss_columns, ax=axes[0], title="Detector losses")
        history.plot(x="epoch", y=metric_columns, ax=axes[1], title="Validation detection accuracy")
        axes[0].set_ylabel("Loss")
        axes[1].set_ylabel("mAP")
        axes[1].set_ylim(0, 1)
        plt.tight_layout()
        plt.show()

In [ ]:
if detector is not None:
    val_names = {r["image"] for r in split["photos"] if r["split"] == "val"}
    diagnostics, previews = [], []
    for record in records:
        if record["image"] not in val_names:
            continue
        result = detector.predict(str(RAW / "images" / record["image"]), imgsz=IMAGE_SIZE,
                                  conf=DETECTION_CONFIDENCE, device=DEVICE, verbose=False)[0]
        predicted = result.obb.xyxyxyxy.cpu().numpy()
        truth = np.asarray(record["boxes"]).reshape(-1, 4, 2) * [record["width"], record["height"]]
        matches = match_boxes(truth, predicted)
        missed, extra = len(truth) - len(matches), len(predicted) - len(matches)
        diagnostics.append({"image": record["image"], "expected": len(truth), "detected": len(predicted), "missed": missed, "extra": extra})
        previews.append((missed + extra, record, predicted, truth))
    frame = pd.DataFrame(diagnostics)
    display(frame.sort_values(["missed", "extra"], ascending=False))
    frame.to_csv(RUN / "validation_tile_counts.csv", index=False)
    for _, record, predicted, truth in sorted(previews, key=lambda x: x[0], reverse=True)[:3]:
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.imshow(read_rgb(RAW / "images" / record["image"]))
        for boxes, color, label in [(truth, "#e6a044", "Labeled"), (predicted, "#367bb1", "Detected")]:
            for i, box in enumerate(boxes):
                p = np.vstack([box, box[0]])
                ax.plot(p[:, 0], p[:, 1], color=color, linewidth=1.5, label=label if i == 0 else None)
        ax.set_title(record["image"])
        ax.legend()
        ax.axis("off")
        plt.show()

## Export and verify mobile models
Export FP32 first, then optionally INT8. INT8 calibration uses **training photos only**.
Compare exports on validation before selecting a default. Keep both versions and measured
sizes. [Current LiteRT export API](https://docs.ultralytics.com/integrations/litert/).

The export bundle is an integration input. The app still needs OBB tensor decoding,
letterbox coordinate reversal, and the identical tile rectification used in notebook 2.
The current SSD app decoder cannot consume these models just by changing a filename.

In [ ]:
if ACTION == "export" and detector is not None:
    export_dir = RUN / "exports"
    export_dir.mkdir(exist_ok=True)
    local_export = WORK / "export" / RUN_NAME
    local_export.mkdir(parents=True, exist_ok=True)
    export_checkpoint = local_export / "domino_detector.pt"
    shutil.copy2(RUN / "weights/best.pt", export_checkpoint)
    export_rows = []
    for mode, quantize in [("fp32", 32)] + ([("int8", 8)] if EXPORT_INT8 else []):
        export_model = YOLO(str(export_checkpoint))
        produced = Path(export_model.export(format="litert", quantize=quantize, imgsz=IMAGE_SIZE,
                                            batch=1, device="cpu", data=str(DATA_YAML.parent / "calibration.yaml")))
        if not produced.is_file() or produced.suffix != ".tflite":
            raise RuntimeError(f"Exporter did not return a .tflite file: {produced}")
        saved = export_dir / f"domino_detector_{mode}.tflite"
        shutil.copy2(produced, saved)
        exported_metrics = YOLO(str(saved), task="obb").val(data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE,
                                                           device="cpu", project=str(RUN / "checks"), name=f"export_{mode}", exist_ok=True)
        score = float(exported_metrics.box.map)
        export_rows.append({"format": mode, "file": saved.name, "bytes": saved.stat().st_size,
                            "map50_95": score, "drop_from_pytorch": val_scores["map50_95"] - score})
    acceptable = [r for r in export_rows if r["drop_from_pytorch"] <= MAX_EXPORT_MAP_DROP]
    preferred = next((r for r in acceptable if r["format"] == "int8"), next((r for r in acceptable if r["format"] == "fp32"), None))
    write_json(export_dir / "detector_metadata.json", {"task": "obb", "classes": ["domino"], "image_size": IMAGE_SIZE,
               "confidence": DETECTION_CONFIDENCE, "split_signature": split["signature"], "crop_version": CROP_VERSION,
               "preferred_file": preferred["file"] if preferred else None, "validation": export_rows,
               "preprocessing": "RGB, preserve aspect ratio with Ultralytics letterbox; invert padding/scale to original pixels",
               "postprocessing": "Read the exported tensor metadata; decode OBB, confidences and classes using the pinned Ultralytics backend. Do not reuse SSD output parsing."})
    display(pd.DataFrame(export_rows))
    if preferred is None:
        raise RuntimeError("No export met the validation tolerance. Keep the checkpoint and investigate conversion before deployment.")
    print("Preferred validated export:", export_dir / preferred["file"])

## Final test and handoff
Enable `FINAL_TEST` only after choosing the model, confidence, and export using validation.
Do not use these test results to repeatedly tune the same experiment. Notebook 2 also
reports full-photo scores once its pip labels and classifier are available.

In [ ]:
if FINAL_TEST and detector is not None:
    test_metrics = detector.val(data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, device=DEVICE,
                                project=str(RUN / "checks"), name="test", exist_ok=True, plots=True)
    write_json(RUN / "test.json", {"map50": float(test_metrics.box.map50), "map50_95": float(test_metrics.box.map)})
    print("Final held-out detection mAP50–95:", float(test_metrics.box.map))
print("Next: open 02_pip_classifier.ipynb with the same DATASET_NAME and Drive project.")
print("Labeling can start before detector training finishes; do not run both notebooks' GPU training at once.")